# Análise Exploratória - ESA Anomaly Dataset

Este notebook faz parte de uma Iniciação Científica que estuda como avaliar a **confiabilidade de alertas em telemetria espacial**.

## Contexto

Satélites geram continuamente séries temporais chamadas de **telemetria** — medidas de temperatura, tensão, corrente e outros parâmetros dos seus subsistemas. Sistemas de monitoramento observam esses dados e emitem **alertas** quando detectam  comportamentos fora do padrão.

O problema central é que **um alerta não significa necessariamente uma falha real**.  O mesmo comportamento atípico pode ser:

- 🔴 Uma **anomalia real** — algo efetivamente errado no satélite
- 🟡 Um **evento nominal raro** — uma operação planejada, incomum mas esperada  (ex: uma manobra, uma calibração)
- ⚪ **Ruído ou problema de qualidade dos dados**

Esta análise exploratória vai nos ajudar a **entender a estrutura dos dados disponíveis** antes de construir qualquer detector ou avaliador de alertas.

In [2]:
# import das bibliotecas e carregamento dos dataset
import pickle
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../../data/ESA/ESA-Mission1")

def load_zip_pickle(path: Path) -> pd.DataFrame:
    """Carrega um DataFrame pickled dentro de um ZIP."""
    with zipfile.ZipFile(path) as z:
        name = z.namelist()[0]
        return pickle.loads(z.read(name))

## O que vamos fazer aqui

Este notebook segue uma progressão natural:

1. **Carregar e inspecionar os metadados** (CSVs) — entender o que existe antes de abrir os arquivos grandes
2. **Carregar as séries temporais** dos canais 41 - 46 — ver como os dados reais se parecem
3. **Visualizar os canais** — identificar padrões, anomalias e eventos nominais raros visualmente
4. **Carregar e visualizar telecomandos** — entender quando e com que frequência comandos foram enviados
5. **Explorar a possibilidade de combinação** — verificar se faz sentido cruzar séries de 
   telemetria com o histórico de telecomandos (join temporal)

## O dataset: ESA-ADB Mission1

Usamos o **ESA Anomaly Detection Benchmark (ESA-ADB)**, um benchmark público  publicado pela Agência Espacial Europeia (ESA) em parceria com a KP Labs.

> 📎 [Artigo original](https://arxiv.org/abs/2406.17826) 

> [Repositório oficial](https://github.com/kplabs-pl/ESA-ADB) 

> [Zenodo (dados)](https://doi.org/10.5281/zenodo.12528696)

### ⚠️ Importante: os dados são anonimizados

Os nomes reais dos canais, subsistemas e telecomandos **não são divulgados**.  Por isso, você verá identificadores genéricos como `channel_41`, `subsystem_5`  e `telecommand_198`. Isso é intencional — protege informações proprietárias da missão.

### O que está disponível neste subconjunto

O ESA-ADB é grande; aqui trabalhamos com um **subconjunto leve** da Mission1:

| Arquivo | O que contém |
|---|---|
| `channels.csv` | Catálogo dos 76 canais: subsistema, unidade física, grupo, flag de alvo |
| `labels.csv` | Eventos rotulados por canal (início, fim, tipo de evento) |
| `anomaly_types.csv` | Tipos de eventos: Anomalia vs. Evento Nominal Raro |
| `telecommands.csv` | Catálogo dos 698 tipos de telecomandos com prioridade |
| `channels/channel_4{1..6}.zip` | Séries temporais dos canais 41 a 46 |
| `telecommands/telecommand_*.zip` | Histórico de quando cada comando foi executado |

### Estrutura dos arquivos de série temporal

Os arquivos `.zip` de canais e telecomandos contêm **DataFrames pickled**  (serializados em formato Python). Cada um tem:

- **Índice:** `datetime` — o timestamp da medição ou execução do comando
- **Coluna:** o valor medido (canal) ou uma flag `1` indicando execução (telecomando)

Os **6 canais disponíveis** (41-46) pertencem todos ao mesmo subsistema e grupo,  o que significa que monitoram aspectos relacionados do satélite.

Os **telecomandos** registram os comandos enviados ao satélite ao longo dos anos. Cada arquivo corresponde a um tipo de comando e armazena todos os momentos em que ele foi executado. Alguns comandos são raros (poucas dezenas de execuções) e outros são frequentes (centenas de milhares).


In [3]:
channels_meta = pd.read_csv(DATA_DIR / "channels.csv")
labels = pd.read_csv(DATA_DIR / "labels.csv", parse_dates=["StartTime", "EndTime"])
anomaly_types = pd.read_csv(DATA_DIR / "anomaly_types.csv")
tc_meta = pd.read_csv(DATA_DIR / "telecommands.csv")


### 📡 Tabela 1: `channels.csv` — "Quais sensores existem no satélite?"

Cada linha representa um **canal de telemetria** — uma série temporal coletada  continuamente pelo satélite.

> Exemplo: Imagine que o satélite é como um avião. Dentro dele existem dezenas de sensores  medindo coisas o tempo todo: temperatura do motor, nível de combustível, pressão  da cabine, etc. No mundo dos satélites, cada um desses sensores é chamado de **canal**. Esta tabela é basicamente o **manual de bordo** — ela lista todos os canais  existentes e diz o que cada um mede.

> 💡 Dos 76 canais, 58 são `Target=YES`. Os 6 canais que temos disponíveis (41–46) são todos do mesmo subsistema e grupo, ou seja, medem grandezas relacionadas entre si.

| Coluna | O que significa |
|---|---|
| `Channel` | Nome identificador do canal (ex: `channel_41`). Anonimizado. |
| `Subsystem` | Subsistema do satélite ao qual o canal pertence. Um subsistema agrupa componentes relacionados (ex: energia, atitude, comunicação). Também anonimizado. |
| `Physical Unit` | Tipo de grandeza física medida (ex: temperatura, tensão, corrente). Anonimizado. |
| `Group` | Agrupamento temático dos canais — canais do mesmo grupo tendem a medir aspectos relacionados. |
| `Target` | `YES` indica que este canal é **alvo da detecção de anomalias**. `NO` significa que o canal existe apenas como contexto e não é avaliado diretamente. |


In [4]:
print("=== channels_meta ===")
display(channels_meta.head())

print(channels_meta['Target'].value_counts())
print('')

print("=== channels_meta (canais 41–46) ===")
canais_disponiveis = [f"channel_{i}" for i in range(41, 47)]
display(channels_meta[channels_meta["Channel"].isin(canais_disponiveis)])

=== channels_meta ===


,Channel,Subsystem,Physical Unit,Group,Target
0,channel_1,subsystem_1,physical_unit_1,1,NO
1,channel_2,subsystem_1,physical_unit_1,1,NO
2,channel_3,subsystem_1,physical_unit_1,1,NO
3,channel_4,subsystem_1,physical_unit_2,2,NO
4,channel_5,subsystem_1,physical_unit_2,2,NO


Target
YES    58
NO     18
Name: count, dtype: int64

=== channels_meta (canais 41–46) ===


,Channel,Subsystem,Physical Unit,Group,Target
40,channel_41,subsystem_5,physical_unit_4,8,YES
41,channel_42,subsystem_5,physical_unit_4,8,YES
42,channel_43,subsystem_5,physical_unit_4,8,YES
43,channel_44,subsystem_5,physical_unit_4,8,YES
44,channel_45,subsystem_5,physical_unit_4,8,YES
45,channel_46,subsystem_5,physical_unit_4,8,YES


### 🏷️ Tabela 2: `labels.csv` — "O que realmente aconteceu, e quando?"

Quando engenheiros da ESA analisaram os dados históricos do satélite, eles identificaram eventos importantes e anotaram: *"entre essa data e essa data, aconteceu alguma coisa nesse canal"*.

Essa tabela é o resultado dessas anotações — o nosso **gabarito oficial**. Ela nos diz o que de fato ocorreu, o que é essencial para avaliar se um detector automático teria acertado ou errado.

| Coluna | Significado simples |
|---|---|
| `ID` | O "nome" do evento. Se o mesmo `ID` aparece em várias linhas (como `id_1` nas 5 primeiras linhas), significa que aquele evento afetou vários canais ao mesmo tempo — não foi só um sensor que reagiu, foram vários juntos. |
| `Channel` | Qual sensor registrou esse evento. |
| `StartTime` | Quando o evento começou. |
| `EndTime` | Quando o evento terminou. |

> ⚠️ **Repare numa coisa:** o `ID` aqui identifica a *ocorrência* — ou seja,  "aconteceu uma vez". Ele **não diz** se foi uma coisa grave ou inofensiva. Para saber isso, precisamos cruzar com a próxima tabela.

> Também podemos notar que algumas linhas têm valores estranhos no `StartTime` (aparece `"Resultado"` em vez de uma data). Isso é um dado ausente ou corrompido — algo que vamos tratar na hora de usar os dados de verdade.


In [5]:
print("=== labels ===")
display(labels.head())

print("=== labels (canais 41–46) ===")
canais_disponiveis = [f"channel_{i}" for i in range(41, 47)]
display(labels[labels["Channel"].isin(canais_disponiveis)])

=== labels ===


,ID,Channel,StartTime,EndTime
0,id_1,channel_12,2004-12-01T20:42:15.429Z,2004-12-08 22:55:45.429000+00:00
1,id_1,channel_13,2004-12-01T20:42:15.429Z,2004-12-08 22:55:45.429000+00:00
2,id_1,channel_14,2004-12-01T20:43:45.429Z,2004-12-02 02:57:15.429000+00:00
3,id_1,channel_15,Resultado,2004-12-02 02:58:45.429000+00:00
4,id_1,channel_16,2004-12-01T20:43:45.429Z,2004-12-16 16:52:30.429000+00:00


=== labels (canais 41–46) ===


,ID,Channel,StartTime,EndTime
29,id_1,channel_41,2004-12-02T03:07:17.646Z,2004-12-08 23:35:55.146000+00:00
30,id_1,channel_42,2004-12-02T03:07:17.646Z,2004-12-08 23:35:55.146000+00:00
31,id_1,channel_43,2004-12-02T03:07:17.646Z,2004-12-08 23:35:55.146000+00:00
32,id_1,channel_44,2004-12-02T03:07:17.646Z,2004-12-08 23:35:55.146000+00:00
33,id_1,channel_45,2004-12-02T03:07:17.646Z,2004-12-08 23:35:55.146000+00:00
...,...,...,...,...
3574,id_200,channel_42,2000-10-10T01:09:58.323Z,2000-10-11 00:11:58.323000+00:00
3575,id_200,channel_43,2000-10-10T00:23:28.323Z,2000-10-11 04:46:58.323000+00:00
3576,id_200,channel_44,2000-10-10T00:40:58.323Z,2000-10-11 03:24:58.323000+00:00
3577,id_200,channel_45,2000-10-10T00:46:58.323Z,2000-10-11 01:15:58.323000+00:00


### 🔍 Tabela 3: `anomaly_types.csv` — "Esse evento foi um problema ou era esperado?"

Nem todo comportamento fora do normal é uma falha. Imagine um avião realizando  uma manobra de pouso: os sensores vão registrar coisas bem diferentes do voo normal — mas isso é *esperado*, não é um problema.

Esta tabela classifica cada tipo de evento em duas categorias principais:

| Coluna | Significado simples |
|---|---|
| `ID` | Liga com o `ID` da tabela anterior — é assim que descobrimos *o que era* aquele evento. |
| `Class` / `Subclass` | Uma classificação técnica mais detalhada do tipo de evento. Os nomes reais foram anonimizados, mas eventos da mesma classe têm características em comum. |
| `Category` | **A coluna mais importante de todas.** Só tem dois valores possíveis: |
| | 🔴 **`Anomaly`** → algo deu errado. É o tipo de evento que um sistema de monitoramento *precisa* detectar. |
| | 🟡 **`Rare Event`** → aconteceu algo incomum, mas que era *esperado* — uma manobra, uma calibração, uma mudança de modo de operação. Não é uma falha, mas parece estranho nos dados. |
| `Locality` | Se o evento foi visível em um trecho pequeno (`Local`) ou em um período longo (`Global`) da série de dados. |
| `Length` | Aqui todos os eventos são `Subsequence` — ou seja, têm começo e fim definidos, não são pontos isolados. |

> 💡 **Por que isso importa para a pesquisa?**  
> Um detector automático simples vai gerar um alerta tanto para uma `Anomaly` quanto para um `Rare Event` — os dois parecem "estranhos" nos dados. O nosso objetivo é entender se é possível usar informações de contexto (como comandos  enviados ao satélite) para ajudar a distinguir um do outro, sem precisar de um especialista humano.


In [6]:
print("=== anomaly_types ===")
display(anomaly_types.head())

=== anomaly_types ===


,ID,Class,Subclass,Category,Dimensionality,Locality,Length
0,id_1,class_6,subclass_1,Rare Event,Multivariate,Global,Subsequence
1,id_2,class_7,subclass_1,Anomaly,Multivariate,Local,Subsequence
2,id_3,class_7,subclass_1,Anomaly,Multivariate,Global,Subsequence
3,id_4,class_7,subclass_1,Anomaly,Multivariate,Local,Subsequence
4,id_5,class_7,subclass_1,Anomaly,Multivariate,Local,Subsequence


### 🛰️ Tabela 4: `tc_meta` (`telecommands.csv`) — "Que comandos podem ser enviados ao satélite?"

O satélite não funciona sozinho. A equipe em terra pode enviar **comandos** para ele — instruções como "ligue esse sensor", "mude a orientação", "entre em modo de economia de energia". Esses comandos se chamam **telecomandos** (TC).

Esta tabela é os **tipos de comandos disponíveis**: lista todos os 698 tipos de telecomandos que existem nessa missão.

| Coluna | Significado simples |
|---|---|
| `Telecommand` | O código do tipo de comando (ex: `telecommand_198`). Anonimizado, assim como os canais. |
| `Priority` | Uma indicação de quão "urgente" ou "importante" esse tipo de comando é. |

> 📌 **O que essa tabela NÃO tem:** ela não diz *quando* cada comando foi usado. 

> O **histórico real de execuções** — ou seja, *"em qual data e hora o da `telecommand_198` foi enviado ao satélite?"* — está nos arquivos `.zip` dentro da pasta `telecommands/`. Vamos carregar esses arquivos nas próximas células para explorar isso.

> A ideia é verificar: se um telecomando foi enviado pouco antes de um sensor registrar algo incomum, isso pode ser uma pista de que o "comportamento estranho" era na verdade uma resposta *esperada* ao comando — e não uma falha.


In [7]:
print("=== tc_meta ===")
display(tc_meta.head(-10))

=== tc_meta ===


,Telecommand,Priority
0,telecommand_1,1
1,telecommand_2,1
2,telecommand_3,1
3,telecommand_4,1
4,telecommand_5,1
...,...,...
683,telecommand_684,1
684,telecommand_685,1
685,telecommand_686,1
686,telecommand_687,1



## 📂 Os dados reais: séries temporais dos canais 41–46

Até agora vimos apenas as **tabelas de metadados** — informações *sobre* os dados. Agora vamos abrir os arquivos que contêm as **leituras reais dos sensores**: o que o satélite mediu, segundo a segundo, ao longo de anos de operação.

### Como esses arquivos funcionam?

Cada canal tem um arquivo `.zip` próprio. Dentro de cada ZIP há um único arquivo no formato **pickle** — um formato do Python para salvar objetos diretamente, sem precisar converter para texto. O resultado, ao carregar, é um `DataFrame` do pandas com esta estrutura:

| O quê | Detalhes |
|---|---|
| **Índice** | Timestamps — data e hora de cada leitura, com precisão de milissegundos |
| **Coluna** | O valor medido pelo sensor naquele instante (número decimal) |

> 💡 Os timestamps **não são igualmente espaçados** — o satélite não mede sempre no mesmo intervalo. Às vezes há leituras com segundos de diferença, às vezes com minutos. Isso é importante de entender antes de fazer qualquer  análise temporal.

> ⚠️ Cada arquivo é grande: o `channel_41`, por exemplo, tem mais de  *15 milhões de linhas** — representando anos de leituras contínuas. Por isso, para visualizar, vamos usar amostras.


In [9]:
# Carregar todos os canais disponíveis
channel_ids = [41, 42, 43, 44, 45, 46]
channels = {}

for cid in channel_ids:
    path = DATA_DIR / "channels" / f"channel_{cid}.zip"
    channels[f"channel_{cid}"] = load_zip_pickle(path)

# Inspecionar o primeiro como exemplo
ch41 = channels["channel_41"]

print("=== Primeiras leituras ===")
display(ch41.head())

print("=== Últimas leituras ===")
display(ch41.tail())

print("=== Resumo estatístico ===")
display(ch41.describe())

print("\n=== Informações gerais de todos os canais ===")
for nome, df in channels.items():
    inicio = df.index.min()
    fim = df.index.max()
    duracao = fim - inicio
    print(f"{nome}: {len(df):>12,.0f} leituras | {inicio.date()} → {fim.date()} | duração: {duracao.days} dias")


/tmp/ipykernel_76631/3751775339.py:15: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  return pickle.loads(z.read(name))


=== Primeiras leituras ===


,channel_41
datetime,
2000-01-01 00:00:16.353,0.812578
2000-01-01 00:00:46.353,0.821213
2000-01-01 00:01:08.853,0.808654
2000-01-01 00:01:16.353,0.819642
2000-01-01 00:01:46.353,0.821996


=== Últimas leituras ===


,channel_41
datetime,
2013-12-31 23:57:39.657,0.806299
2013-12-31 23:58:09.657,0.806299
2013-12-31 23:58:39.657,0.806299
2013-12-31 23:59:09.657,0.806299
2013-12-31 23:59:39.657,0.808654


=== Resumo estatístico ===


,channel_41
count,1.538117e+07
mean,8.096232e-01
std,1.565005e-02
min,0.000000e+00
25%,8.055143e-01
50%,8.094388e-01
75%,8.141486e-01
max,9.821104e-01



=== Informações gerais de todos os canais ===
channel_41:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias
channel_42:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias
channel_43:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias
channel_44:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias
channel_45:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias
channel_46:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias


### Interpretando os Dados Acima

#### A coluna `channel_4*` — o que são esses números?

A coluna recebe o nome do canal e armazena o **valor medido pelo sensor** naquele instante. Como os dados são anonimizados, não sabemos a grandeza física real (temperatura? tensão? pressão?) nem a unidade de medida.

O que importa é analisarmos  **como ele muda ao longo do tempo**: uma leitura muito diferente do padrão histórico é o que nos 
interessa detectar.


#### Primeiras leituras — início da missão

O sensor começou a ser monitorado em **1º de janeiro de 2000**. Nas primeiras  eituras, os valores ficam em torno de **0.81–0.82** e o intervalo entre medições varia: ora 30 segundos, ora 22 segundos, ora 7 segundos. Isso confirma que a frequência de amostragem **não é fixa** — o satélite não mede sempre no mesmo ritmo.


#### Resumo estatístico

| O que olhar | O que significa neste canal |
|---|---|
| **Média ≈ 0.810** | O valor "normal" que o sensor costuma registrar |
| **Desvio padrão ≈ 0.016** | O sensor é bem estável — a variação típica é de apenas ~2% em torno da média |
| **Mínimo = 0.000** | ⚠️ Pelo menos uma vez o sensor registrou zero — muito abaixo do padrão. Pode ser uma anomalia, sensor desligado ou dado corrompido |
| **Máximo ≈ 0.982** | O maior valor já registrado — ainda dentro de um range razoável, mas ~21% acima da média |
| **Percentis 25–75%: 0.805 a 0.814** | Em 50% do tempo, o sensor fica numa faixa estreita de apenas 0.009 de amplitude — confirma a estabilidade |


#### Todos os canais têm exatamente a mesma quantidade de leituras

Os 6 canais têm **15.381.169 leituras cada**, do mesmo período (2000–2013). Isso não é coincidência — como pertencem ao mesmo subsistema e grupo, são coletados de forma sincronizada. 

### 🛰️ Como os arquivos de telecomandos são organizados?


A pasta `telecommands/` tem um arquivo `.zip` para **cada tipo de telecomando** — ou seja, 698 arquivos no total.

> Imagine que cada arquivo é como um **diário de bordo de um botão específico do painel de controle**: ele registra todas as vezes que aquele botão foi pressionado, com a data e hora exata.

Cada arquivo `.zip` contém um DataFrame pickled com a seguinte estrutura:

| O quê | Detalhes |
|---|---|
| **Índice** | `datetime` — o timestamp de quando o comando foi executado |
| **Coluna** | O valor `1` em todas as linhas — indica simplesmente que o comando *ocorreu* naquele instante |

> **Atenção:** ao contrário dos canais (que armazenam valores numéricos contínuos), os arquivos de telecomandos não medem grandezas — eles registram *eventos*. O valor `1` é apenas uma marcação: "sim, isso aconteceu".

> 💡 **Por que isso importa?** - Hipótese: Se um telecomando foi enviado pouco antes de um sensor registrar algo estranho, isso pode indicar que o "comportamento estranho" era na verdade uma *resposta esperada* ao comando — e não uma falha real. 

In [10]:
# Carregar alguns telecomandos como exemplo
# Usamos o tc_198 (maior arquivo = mais execuções) e alguns pequenos para comparar
tcs_para_explorar = [198, 270, 482, 10]

tc_series = {}
for tc_id in tcs_para_explorar:
    path = DATA_DIR / "telecommands" / f"telecommand_{tc_id}.zip"
    tc_series[f"telecommand_{tc_id}"] = load_zip_pickle(path)

# Inspecionar o telecommand_198 como exemplo (o mais frequente)
tc198 = tc_series["telecommand_198"]

print("=== Primeiras execuções do telecommand_198 ===")
display(tc198.head())

print("=== Últimas execuções ===")
display(tc198.tail())

print("\n=== Informações gerais dos telecomandos carregados ===")
for nome, df in tc_series.items():
    inicio = df.index.min()
    fim = df.index.max()
    duracao = fim - inicio
    print(f"{nome}: {len(df):>10,.0f} execuções | {inicio.date()} → {fim.date()} | duração: {duracao.days} dias")


=== Primeiras execuções do telecommand_198 ===


/tmp/ipykernel_76631/3751775339.py:15: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  return pickle.loads(z.read(name))


,telecommand_198
2000-01-03 07:29:27.549,1
2000-01-03 07:29:29.337,1
2000-01-03 07:29:30.993,1
2000-01-03 07:29:33.024,1
2000-01-03 07:29:34.824,1


=== Últimas execuções ===


,telecommand_198
2013-12-31 16:21:06.300,1
2013-12-31 16:21:08.106,1
2013-12-31 16:21:09.915,1
2013-12-31 16:21:11.742,1
2013-12-31 16:21:13.515,1



=== Informações gerais dos telecomandos carregados ===
telecommand_198:    266,724 execuções | 2000-01-03 → 2013-12-31 | duração: 5111 dias
telecommand_270:    327,502 execuções | 2000-01-03 → 2013-12-31 | duração: 5111 dias
telecommand_482:    519,992 execuções | 2000-02-26 → 2013-04-25 | duração: 4807 dias
telecommand_10:        897 execuções | 2000-02-10 → 2013-09-12 | duração: 4962 dias


### Interpretando os dados acima

#### O que a diferença de tamanho dos arquivos nos diz?

| Telecomando | Tamanho aprox. | O que isso sugere |
|---|---|---|
| `telecommand_198` | ~1.2 MB | Executado com muita frequência |
| `telecommand_270` | ~1.4 MB | Também muito frequente |
| `telecommand_10` | ~5 KB | Raro — executado poucas vezes ao longo de anos |

> ⚠️ **Cuidado metodológico importante:** saber que um telecomando foi executado **não prova** que ele causou uma mudança no sensor. Os dados são anonimizados — não sabemos o que cada comando faz fisicamente. O que podemos observar é **proximidade temporal**, não causalidade.


## 🧹 Limpeza e ajustes nos dados

Antes de fazer qualquer análise, precisamos garantir que os dados limpos. Essa etapa **não inventa nem remove informação** — ela corrige inconsistências de formato para que o Python consiga trabalhar corretamente.

### O que será ajustado:

| Problema encontrado | Onde | O que faremos |
|---|---|---|
| Timestamps com valor `"Resultado"` (texto em vez de data) | `labels.csv` → coluna `StartTime` | Converter para `NaT` (data inválida) e remover essas linhas |
| Timestamps sem timezone definida | `labels.csv`, canais e telecomandos | Padronizar tudo para UTC |
| Índice duplicado nos canais | séries temporais `.zip` | Verificar e remover duplicatas |
| Valor zero nos canais (mínimo = 0.0 visto no `describe()`) | `channels/` | Documentar — **não remover** |

> ⚠️ **Regra de ouro desta pesquisa:** nunca alteramos os valores originais das
> séries temporais. Zeros e leituras estranhas podem ser anomalias reais que
> queremos detectar — removê-los distorceria o estudo. O que fazemos é
> **anotar** que existem e tratar apenas o que é claramente um problema de
> formato (datas inválidas, timezones misturadas).

### ⚙️ O único ajuste obrigatório: padronização de timezone

Existe **um problema técnico** que precisa ser corrigido para que as análises
funcionem — não é um problema nos dados, mas na forma como o Python os lê.

**Situação atual:**
- `labels.csv` → timestamps com timezone **UTC** (ex: `2004-12-01 20:42:15+00:00`)
- `channels/*.zip` → timestamps **sem timezone** (ex: `2000-01-01 00:00:16`)

Quando tentamos comparar um timestamp do labels com um timestamp de um canal,
o Python retorna erro: `TypeError: Cannot compare tz-naive and tz-aware datetime-like objects`

**O que fazemos:** declaramos que os timestamps dos canais e telecomandos
são UTC usando `tz_localize("UTC")`. Isso **não muda nenhum valor** — apenas adiciona a informação de fuso que estava implícita nos dados originais e ajusta o datatype deles para ser datetime padrão.



In [16]:
# ─── AJUSTE: padronizar timezone para UTC ────────────────────────────────────
#
# tz_localize("UTC") declara o fuso horário dos timestamps.
# Só aplicamos se o índice ainda não tiver timezone (tz is None).
# Isso modifica o índice in-place nos dicionários channels e tc_series.
#
# NÃO altera nenhum valor de leitura ou execução — apenas adiciona
# a informação de fuso que estava implícita.

print("=== Padronizando timezone dos canais ===")
for nome, df in channels.items():
    if df.index.tz is None:
        channels[nome].index = df.index.tz_localize("UTC")
        print(f"  {nome}: sem tz → UTC ✅")
    else:
        print(f"  {nome}: já é {df.index.tz} ✅")

print("\n=== Padronizando timezone dos telecomandos ===")
for nome, df in tc_series.items():
    if df.index.tz is None:
        tc_series[nome].index = df.index.tz_localize("UTC")
        print(f"  {nome}: sem tz → UTC ✅")
    else:
        print(f"  {nome}: já é {df.index.tz} ✅")


=== Padronizando timezone dos canais ===
  channel_41: sem tz → UTC ✅
  channel_42: sem tz → UTC ✅
  channel_43: sem tz → UTC ✅
  channel_44: sem tz → UTC ✅
  channel_45: sem tz → UTC ✅
  channel_46: sem tz → UTC ✅

=== Padronizando timezone dos telecomandos ===
  telecommand_198: sem tz → UTC ✅
  telecommand_270: sem tz → UTC ✅
  telecommand_482: sem tz → UTC ✅
  telecommand_10: sem tz → UTC ✅


# Tabela `labels.csv`

Identificamos que a coluna StartTime contém algumas entradas com o texto "Resultado" em vez de uma data. Vamos contar e inspecionar sem remover nada. 

> pd.to_datetime com errors="coerce" tenta converter cada valor para datetime. Se não conseguir (como com "Resultado"), retorna NaT (Not a Time). 

In [12]:
start_convertido = pd.to_datetime(labels["StartTime"], utc=True, errors="coerce")
end_convertido   = pd.to_datetime(labels["EndTime"],   utc=True, errors="coerce")

n_start_invalido = start_convertido.isna().sum()
n_end_invalido   = end_convertido.isna().sum()

print("=== Diagnóstico: labels.csv ===")
print(f"Total de linhas:                    {len(labels):,}")
print(f"StartTime com valor não-data (NaT): {n_start_invalido}")
print(f"EndTime com valor não-data (NaT):   {n_end_invalido}")

# Mostrar as linhas com esse problema para entender o padrão
if n_start_invalido > 0:
    linhas_problematicas = labels[start_convertido.isna()]
    print(f"\nLinhas com StartTime inválido:")
    display(linhas_problematicas.head(10))
    print(f"\n⚠️  Essas linhas existem nos dados originais e são mantidas.")
    print(f"   Nas análises que exigem data de início, usaremos uma cópia filtrada.")

=== Diagnóstico: labels.csv ===
Total de linhas:                    3,589
StartTime com valor não-data (NaT): 1
EndTime com valor não-data (NaT):   0

Linhas com StartTime inválido:


,ID,Channel,StartTime,EndTime
3,id_1,channel_15,Resultado,2004-12-02 02:58:45.429000+00:00



⚠️  Essas linhas existem nos dados originais e são mantidas.
   Nas análises que exigem data de início, usaremos uma cópia filtrada.


In [17]:
# ─── VARIÁVEL DE TRABALHO: labels com datas válidas ──────────────────────────
#
# O labels original é mantido intacto.
#
# Para análises que exigem comparação de datas (como calcular duração de eventos
# ou fazer join temporal com os canais), precisamos de uma versão onde:
#   1. StartTime e EndTime estejam em formato datetime com UTC
#   2. Linhas com datas inválidas sejam excluídas NESSA CÓPIA APENAS
#
# Nomeamos com sufixo "_clean" para deixar claro que é uma cópia de trabalho.

labels_clean = labels.copy()
labels_clean["StartTime"] = pd.to_datetime(labels_clean["StartTime"], utc=True, errors="coerce")
labels_clean["EndTime"]   = pd.to_datetime(labels_clean["EndTime"],   utc=True, errors="coerce")
labels_clean = labels_clean.dropna(subset=["StartTime", "EndTime"])

print("=== labels original (intacto) ===")
print(f"  Linhas: {len(labels):,}")
print(f"  Tipo StartTime: {labels['StartTime'].dtype}")

print("\n=== labels_clean (cópia de trabalho) ===")
print(f"  Linhas: {len(labels_clean):,}")
print(f"  Linhas excluídas desta cópia: {len(labels) - len(labels_clean)}")
print(f"  Tipo StartTime: {labels_clean['StartTime'].dtype}")

print("\n✅ labels original preservado. Nas análises temporais, usaremos labels_clean.")


=== labels original (intacto) ===
  Linhas: 3,589
  Tipo StartTime: str

=== labels_clean (cópia de trabalho) ===
  Linhas: 3,588
  Linhas excluídas desta cópia: 1
  Tipo StartTime: datetime64[us, UTC]

✅ labels original preservado. Nas análises temporais, usaremos labels_clean.


# Canais de Telemetria

Verificamos dois tipos de situações nos canais:
- Timestamps duplicados — verificar se há datas duplicadas 
- Valores zero — muito abaixo do padrão (~0.81) observado no describe()

Em ambos os casos: APENAS DOCUMENTAMOS. Não removemos.

In [14]:
print("=== Diagnóstico: canais de telemetria ===\n")
for nome, df in channels.items():
    serie = df.squeeze()  # transformar DataFrame de 1 coluna em Series
    
    n_duplicatas = df.index.duplicated().sum()
    n_zeros      = (serie == 0).sum()
    pct_zeros    = n_zeros / len(serie) * 100
    
    flag_dup  = "⚠️ " if n_duplicatas > 0 else "✅"
    flag_zero = "⚠️ " if n_zeros > 0 else "✅"
    
    print(f"  {nome}:")
    print(f"    {flag_dup}  Timestamps duplicados: {n_duplicatas:,}")
    print(f"    {flag_zero}  Leituras com valor zero: {n_zeros:,} ({pct_zeros:.4f}% do total)")
    print()

=== Diagnóstico: canais de telemetria ===

  channel_41:
    ✅  Timestamps duplicados: 0
    ⚠️   Leituras com valor zero: 2,940 (0.0191% do total)

  channel_42:
    ✅  Timestamps duplicados: 0
    ⚠️   Leituras com valor zero: 2,938 (0.0191% do total)

  channel_43:
    ✅  Timestamps duplicados: 0
    ⚠️   Leituras com valor zero: 5,134 (0.0334% do total)

  channel_44:
    ✅  Timestamps duplicados: 0
    ⚠️   Leituras com valor zero: 8,874 (0.0577% do total)

  channel_45:
    ✅  Timestamps duplicados: 0
    ⚠️   Leituras com valor zero: 2,939 (0.0191% do total)

  channel_46:
    ✅  Timestamps duplicados: 0
    ⚠️   Leituras com valor zero: 2,942 (0.0191% do total)



### ✅ Estado dos dados após esta seção

| Dado | Alterado? | O que mudou |
|---|---|---|
| `labels` | ❌ Não | Original intacto |
| `labels_clean` | Cópia | Timestamps em UTC, linhas com data inválida excluídas *desta cópia* |
| `channels` | Só o índice | Timezone adicionado (sem alterar valores) |
| `tc_series` | Só o índice | Timezone adicionado (sem alterar valores) |

> 💡 Toda decisão tomada aqui deve aparecer na seção de limitações do relatório
> final — especialmente a existência de timestamps inválidos, cujo significado
> real não é conhecido.


# Perguntas que iremos direcionar a análise:

#### 1 Entendendo como são os eventos considerados raros, anomalias e normais

**1.1 Quantos eventos existem nos canais 41–46? Desses, quantos são Anomaly e quantos são Rare Event?**

**1.2 Qual é a duração típica de uma anomalia? E de um evento nominal raro? Elas são mais longas ou mais curtas?**



**1.3 Existem eventos que afetam vários canais ao mesmo tempo (mesmo ID em múltiplos canais)? Com que frequência isso acontece?**

**1.4 Os eventos estão distribuídos uniformemente ao longo do tempo (2000–2013) ou existem períodos com mais ocorrências?**

#### 2. Comportamento dos canais durante eventos

**2.1 Como o valor do channel_41 se comporta durante uma anomalia anotada comparado com um período normal? O desvio é óbvio visualmente?**

**2.2 O mesmo evento aparece nos canais 41–46 ao mesmo tempo ou cada canal reage de forma diferente (em tempos diferentes)?**

**2.3 Existem períodos onde o sensor fica com o mesmo valor por muito tempo (valor "congelado")? Isso pode ser dado ruim, travamento ou sensor em modo de espera?**

**2.4 Qual é o intervalo típico entre leituras (frequência de amostragem)? Existem lacunas grandes - períodos sem nenhuma leitura?**

#### 3. Telecomandos e contexto operacinal

**3.1 Quantos telecomandos diferentes foram executados ao longo dos 14 anos? Quais são os mais frequentes?**

**3.2 Existe algum telecomando que aparece frequentemente nas horas ou dias que antecedem eventos nominais raros nos canais 41–46?**

**3.3 Esse mesmo telecomando também aparece antes de anomalias reais?**

**3.4 Qual é a prioridade dos telecomandos que aparecem próximos aos eventos? Comandos de prioridade 3 são mais associados a eventos do que comandos de prioridade 1?**

#### 4. Qualidade dos Dados

**4.1 Em que proporção do tempo os canais têm dados válidos? Existe algum canal com muito mais lacunas que os outros?**

**4.2 Durante as anomalias anotadas, os dados estavam completos ou havia lacunas/congelamentos que poderiam ter causado o alerta?**

**4.3 Existem valores extremos (muito acima ou abaixo do padrão histórico) que ocorrem fora de qualquer evento anotado? O que isso significa?**

#### 5. Detecção de Anomalias com Limites

**5.1 Se usarmos uma regra simples ("alerta quando o valor sai de 3 desvios padrões da mediana"), quantos alertas seriam gerados? Quantos coincidiriam com anomalias reais? Quantos seriam eventos nominais raros?**

**5.2 Um detector simples conseguiria distinguir os 118 eventos da Mission1 apenas pelo score numérico, sem contexto?**